# VEREC — train a crime action model on Colab

Pose extraction is CPU-bound and slow; training is GPU-bound and fast. The
practical split is: **extract skeletons locally**, upload the small `.npz`
clips, train here.

A few hundred clips compress to a handful of MB, so this avoids uploading
video entirely.

**Runtime → Change runtime type → GPU** before running.

In [ ]:
!nvidia-smi
import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())

## 1. Get the code

Either clone your repo, or upload `action/` and `training/` plus the NTU60
checkpoint. Only those two packages are needed — the FastAPI backend and
frontend are not.

In [ ]:
# Option A — from Google Drive (recommended: survives runtime restarts)
from google.colab import drive
drive.mount('/content/drive')

PROJECT = '/content/drive/MyDrive/verec'   # adjust to taste

import os, sys
os.makedirs(PROJECT, exist_ok=True)
os.chdir(PROJECT)
sys.path.insert(0, PROJECT)
print('working in', os.getcwd())
print(os.listdir('.'))

In [ ]:
# Option B — upload a zip of action/ + training/ + checkpoints/ + clips
# from google.colab import files
# up = files.upload()
# !unzip -q -o {list(up)[0]}

## 2. Sanity-check the clips

Class balance decides whether training is worth starting. If `normal` is under
a third of the set, or a crime class has fewer than ~30 clips, fix the data
before spending GPU time.

In [ ]:
from training.dataset import SkeletonClipDataset
from training.labels import CRIME_ACTIONS

CLIPS = 'training/data/clips'

for split in ('train', 'val', 'test'):
    try:
        ds = SkeletonClipDataset(CLIPS, split)
    except FileNotFoundError:
        print(f'{split}: (missing)')
        continue
    counts = ds.class_counts()
    total = counts.sum()
    print(f'{split}: {total} clips')
    for name, n in zip(CRIME_ACTIONS, counts):
        if n:
            print(f'   {name:16s} {n:5d}  ({n/total:.0%})')

In [ ]:
# Confirm a sample has the shape the model expects: (M, T, V, C)
x, y = ds[0]
print('sample', tuple(x.shape), '-> label', CRIME_ACTIONS[y])
print('value range', float(x[..., :2].min()), 'to', float(x[..., :2].max()))

## 3. Train

`--balance weights` matters here: crime datasets skew hard toward `normal`,
and unweighted training converges to predicting the majority class while
reporting a flattering accuracy.

In [ ]:
!python -m training.train \
    --clips training/data/clips \
    --init checkpoints/stgcn_ntu60_joint.pth \
    --out checkpoints/stgcn_crime.pth \
    --epochs 60 --batch-size 32 --lr 1e-3 \
    --balance weights --early-stop 12 --device cuda

## 4. Evaluate on the held-out test split

Read the confusion matrix, not the accuracy. The line to watch is the rate at
which `normal` clips get flagged as crime — that is what determines whether
anyone can stand to use the alerts.

In [ ]:
!python -m training.evaluate \
    --checkpoint checkpoints/stgcn_crime.pth \
    --clips training/data/clips --split test --device cuda

## 5. Plot training history

In [ ]:
import json
import matplotlib.pyplot as plt

hist = json.load(open('checkpoints/stgcn_crime_history.json'))
ep = [h['epoch'] for h in hist]

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(ep, [h['train_loss'] for h in hist], label='train loss')
if 'loss' in hist[0]:
    ax[0].plot(ep, [h.get('loss') for h in hist], label='val loss')
ax[0].set_xlabel('epoch'); ax[0].legend(); ax[0].set_title('loss')

ax[1].plot(ep, [h['train_acc'] for h in hist], label='train acc')
if 'macro_f1' in hist[0]:
    ax[1].plot(ep, [h.get('macro_f1') for h in hist], label='val macro-F1')
ax[1].set_xlabel('epoch'); ax[1].legend(); ax[1].set_title('accuracy / macro-F1')
plt.tight_layout(); plt.show()

## 6. Download the checkpoint

Take **both** files — the runtime reads class names from the sidecar.

In [ ]:
from google.colab import files
files.download('checkpoints/stgcn_crime.pth')
files.download('checkpoints/stgcn_crime.pth.labels.json')

Then locally:

```bash
export ACTION_MODEL_PATH=checkpoints/stgcn_crime.pth
./dev.sh
```